# The Climate is Changing: How is Our Sentiment?

## Exploring Changing Sentiment in Climate Articles: 2013 to 2023

#### This is a scratch notebook used to organize EDA thoughts. For final EDA and table creation, see create_tables.ipynb

## Corpus Description

92 scientific articles are drawn from PeerJ's publications in the decade spanning 2013 to 2023 (18 from 2013, 37 from 2018, and 37 from 2023). Articles are sorted by PeerJ's determination of relevance to the query "climate change." The top most relevant articles per year, after meeting these described criteria, have then been chosen to comprise my corpus.

## Imports

In [1]:
# Imports
import pandas as pd 
import numpy as np 
import glob
from lxml import etree
import os
import sys

# ----File Stitching----
# If not in repo home folder, cd back 
if os.path.basename(os.getcwd()) != "evolving_sentiment_climate":
    os.chdir('..')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), '..', 'sources'))
source_dir = "sources"
source_files_paths = glob.glob(f"{source_dir}/*.xml")

## Creating DOC Table

In [2]:
# Iterate through files, grab XML info, and save to a dictionary
docs = []
for i, source_file_path in enumerate(source_files_paths):

    # print(source_file_path)

    # Capture tree, root
    tree = etree.parse(source_file_path)
    root = tree.getroot()
    
    # Capture publication id
    pub_id_el = root.xpath("//front//article-meta//article-id")[1]
    pub_id_str = pub_id_el.text
    # print(pub_id_str)

    # Capture title
    title_el = root.xpath("//front//article-title")[0]
    title_str = " ".join([t.strip() for t in title_el.itertext()])
    # print(i, title_str)

    # Capture temporal elements
    year_el = root.xpath("//front//pub-date/year")[0]
    year_str = year_el.text
    month_el = root.xpath("//front//pub-date/month")[0]
    month_str = month_el.text
    day_el = root.xpath("//front//pub-date/day")[0]
    day_str = day_el.text
    date_str = "-".join([year_str, month_str, day_str])
    # print(date_str)

    # Capture key words
    kwd_els = root.xpath("//kwd")
    kwd_strs = [kwd_el.text for kwd_el in kwd_els]
    # print(kwd_strs)

    # Capture each p section (part of OHCO) within the body
    p_els = root.xpath("//body//p")
    p_strs = []
    for p_el in p_els:
        etree.strip_elements(p_el, "xref", with_tail=False)
        p_str = etree.tostring(p_el, method="text", encoding="unicode")
        p_strs.append(p_str)
    # print(p_strs)

    # Add to dictionary
    docs.append({
        'doc_id': pub_id_str,
        'doc_title': title_str,
        'doc_date': date_str,
        'doc_content': " ".join(p_strs),
        'doc_kws': kwd_strs,
        'doc_file_path': source_file_path
    })

# Convert dictionary to data frame
DOC = pd.DataFrame(docs) 
del(docs)

In [3]:
DOC

,doc_id,doc_title,doc_date,doc_content,doc_kws,doc_file_path
0,10.7717/peerj.6130,Local persistence of Mann’s soft-haired mouse ...,2018-12-18,The southern cone of South America has been th...,"[Lowland coastal refugium, Pleistocene, Patago...",sources\Abrothrix_manni_2018.xml
1,10.7717/peerj.16574,Acute heat priming promotes short-term climate...,2023-12-5,Anthropogenic ocean warming threatens the surv...,"[None, Hormetic priming, Environmental memory,...",sources\Acute_heat_2023.xml
2,10.7717/peerj.6133,Effects of ultraviolet radiation on metabolic ...,2018-12-18,"Environmental changes (e.g., global warming, c...","[Metabolic rate, Microbial communities, UV-B r...",sources\Aedes_albopictus_2018.xml
3,10.7717/peerj.5801,Agrichemicals and antibiotics in combination i...,2018-10-12,Neither reducing the use of antibiotics nor di...,"[Antibiotic resistance, Herbicide, Minimum sel...",sources\Agrichemicals_2018.xml
4,10.7717/peerj.188,Transformative optimisation of agricultural la...,2013-10-24,"By 2050, the global human population will have...","[Food security, Deforestation, Climate change,...",sources\Agricultural_2013.xml
...,...,...,...,...,...,...
87,10.7717/peerj.16622,Living on the edge: urban fireflies (Coleopter...,2023-12-14,Urban areas are geographical barriers for biod...,"[Urbanization, Local extinction, Nocturnal ins...",sources\Urban_fireflies_2023.xml
88,10.7717/peerj.16337,Temperature vegetation dryness index (TVDI) fo...,2023-12-18,"In the wake of rising global temperatures, the...","[Drought monitoring, Temperature vegetation dr...",sources\Vegetation_dryness_2023.xml
89,10.7717/peerj.5949,An emerging viral pathogen truncates populatio...,2018-11-16,The emergence of infectious diseases can trunc...,"[Demography, Environmental stochasticity, Dise...",sources\Viral_pathogen_2018.xml
90,10.7717/peerj.16525,Water-use characteristics of Syzygium antisept...,2023-11-30,Over an annual timescale with negligible chang...,"[Tree water use, Secondary forests, None, None...",sources\Water_use_2023.xml


## Creating TOKEN Table

In [4]:
TOKEN = DOC.doc_content.str.split(expand=True).stack().to_frame("token_str")
TOKEN.index.names = ['doc_id','token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z]+", "", regex=True)
TOKEN = TOKEN[~(TOKEN.term_str == "")]
TOKEN

token_str  term_str
doc_id token_num                    
0      0               The       the
       1          southern  southern
       2              cone      cone
       3                of        of
       4             South     south
...                    ...       ...
91     17424           NaN       NaN
       17425           NaN       NaN
       17426           NaN       NaN
       17427           NaN       NaN
       17428           NaN       NaN

[1574428 rows x 2 columns]